# Task 1 — Ingest Data

**Purpose:** first task in our multi-task Workflow. Reads a built-in sample table, filters it, and writes the result to a table that the next task depends on.

**Concept demoed:** Job **Parameters** — this notebook reads values passed in from the Job config via `dbutils.widgets`, instead of having anything hardcoded.

In [ ]:
# These widgets let us run this notebook standalone (fill in manually)
# OR let a Databricks Job pass values in automatically as Job Parameters.
dbutils.widgets.text("catalog", "main", "Target catalog")
dbutils.widgets.text("schema", "default", "Target schema")
dbutils.widgets.text("min_trip_distance", "1.0", "Min trip distance filter")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
min_trip_distance = float(dbutils.widgets.get("min_trip_distance"))

print(f"Running with catalog={catalog}, schema={schema}, min_trip_distance={min_trip_distance}")

In [ ]:
# samples.nyctaxi.trips is a built-in sample dataset available in every
# Databricks workspace (including Free Edition) - no external storage needed.
df = spark.table("samples.nyctaxi.trips")

filtered = df.filter(df.trip_distance >= min_trip_distance)

print(f"Source rows: {df.count()}")
print(f"Filtered rows: {filtered.count()}")
display(filtered.limit(10))

In [ ]:
target_table = f"{catalog}.{schema}.demo_trips_ingested"
filtered.write.mode("overwrite").saveAsTable(target_table)
print(f"Wrote {filtered.count()} rows to {target_table}")

# Hand the table name to the next task via a task value
dbutils.jobs.taskValues.set(key="ingested_table", value=target_table)

**Talking point for the team:** `dbutils.jobs.taskValues.set/get` is how tasks pass small pieces of data to each other in a multi-task Job (e.g., a table name, a row count, a status flag) without writing to disk. This is the Databricks-native equivalent of Airflow's XComs.